# Iteration 6 Model Visualization

Visualize predictions from the best model (TverskyLoss, 46% Extreme recall)

Shows:
- Pre-fire RGB and Post-fire RGB images
- Ground truth burn severity label
- Model-predicted burn severity label

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import json

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Color scheme for 7 burn severity classes
CLASS_COLORS = {
    0: np.array([0.0, 0.5, 0.0]),      # Unburned: Dark Green
    1: np.array([1.0, 1.0, 0.0]),      # Low Severity: Yellow
    2: np.array([1.0, 0.65, 0.0]),     # Moderate: Orange
    3: np.array([1.0, 0.27, 0.0]),     # High: Orange-Red
    4: np.array([0.75, 0.0, 0.0]),     # Extreme: Dark Red
    5: np.array([0.0, 0.4, 1.0]),      # Water: Blue
    6: np.array([0.9, 0.9, 0.9]),      # Cloud: Light Gray
}

CLASS_NAMES = {
    0: 'Unburned',
    1: 'Low Severity',
    2: 'Moderate',
    3: 'High',
    4: 'Extreme',
    5: 'Water',
    6: 'Cloud',
}

print(f"\nClass color scheme:")
for class_id in range(7):
    color = CLASS_COLORS[class_id]
    name = CLASS_NAMES[class_id]
    print(f"  {class_id}: {name:15s} {color}")

In [ ]:
# Load Phase II_01 data
print(f"Loading Phase II_01 outputs...\n")

labels_dir = Path('/content/drive/MyDrive/RETINNA_cache/phase2/II_01_relabeling')

# Find latest files
label_files = sorted(labels_dir.glob('multi_class_labels_*.pt'))
pre_rgbn_files = sorted(labels_dir.glob('pre_rgbn_*.pt'))
post_rgbn_files = sorted(labels_dir.glob('post_rgbn_*.pt'))

latest_labels_file = label_files[-1]
latest_pre_rgbn_file = pre_rgbn_files[-1]
latest_post_rgbn_file = post_rgbn_files[-1]

print(f"Loading labels: {latest_labels_file.name}")
labels_tensor = torch.load(latest_labels_file)
print(f"  Shape: {labels_tensor.shape}")

print(f"Loading pre-fire RGBN: {latest_pre_rgbn_file.name}")
pre_rgbn_tensor = torch.load(latest_pre_rgbn_file)
print(f"  Shape: {pre_rgbn_tensor.shape}")

print(f"Loading post-fire RGBN: {latest_post_rgbn_file.name}")
post_rgbn_tensor = torch.load(latest_post_rgbn_file)
print(f"  Shape: {post_rgbn_tensor.shape}")

# Load metadata for fold information
metadata_paths = [
    Path('/content/drive/MyDrive/RETINNA_DATA/metadata/cabuaur_metadata.json'),
    Path('/content/RETINNA/docs/cabuaur_metadata.json'),
]

metadata_path = None
for path in metadata_paths:
    if path.exists():
        metadata_path = path
        print(f"\nFound metadata at {path}")
        break

print(f"\n✓ Phase II_01 data loaded")

In [ ]:
# Find all model checkpoints
print(f"Looking for model checkpoints...\\n")

checkpoint_dir = Path('/content/drive/MyDrive/RETINNA_cache/phase2/II_02_training')
checkpoint_files = sorted(checkpoint_dir.glob('unet_model_*.pt'))

print(f"Found {len(checkpoint_files)} checkpoints:")
for ckpt in checkpoint_files:
    print(f"  - {ckpt.name}")

if not checkpoint_files:
    print(f"❌ No checkpoints found at {checkpoint_dir}")
else:
    print(f"\\n✓ Will visualize predictions from all checkpoints")

In [ ]:
# U-Net architecture (same as training)
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels // 2, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        if x2.size() != x1.size():
            x1 = F.pad(x1, (0, x2.size(3) - x1.size(3), 0, x2.size(2) - x1.size(2)))
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=8, out_channels=7, bilinear=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.bilinear = bilinear
        
        factor = 2 if bilinear else 1
        
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // factor)
        
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        x = self.outc(x)
        return x

# Create and load model
model = UNet(in_channels=8, out_channels=7, bilinear=True).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load normalization statistics
channel_means = checkpoint['normalization']['channel_mean'].to(device)
channel_stds = checkpoint['normalization']['channel_std'].to(device)

print(f"✓ Model loaded and ready for inference")
print(f"  Channel means: {channel_means}")
print(f"  Channel stds: {channel_stds}")

In [ ]:
# Debug: Check prediction distributions across all checkpoints
print(f"Debugging predictions across all checkpoints...\\n")

for sample_idx in sample_indices[:1]:  # Just check first sample
    gt = all_results[list(all_results.keys())[0]][sample_idx]['ground_truth'].numpy().flatten()
    
    print(f"Sample {sample_idx}:")
    print(f"\\nGround Truth class distribution:")
    for class_id in range(7):
        count = (gt == class_id).sum()
        pct = 100 * count / len(gt)
        print(f"  Class {class_id} ({CLASS_NAMES[class_id]:15s}): {count:6d} ({pct:5.1f}%)")
    
    print(f"\\nPredicted class distribution by checkpoint:")
    print(f"{'Checkpoint':20s} | Unburned | Low Sev | Moderate | High | Extreme | Water | Cloud")
    print(f"{'-' * 20}|{'-' * 80}")
    
    for ckpt_name in sorted(all_results.keys()):
        pred = all_results[ckpt_name][sample_idx]['predictions'].cpu().numpy().flatten()
        dist = [100 * (pred == class_id).sum() / len(pred) for class_id in range(7)]
        print(f"{ckpt_name:20s} | {dist[0]:6.1f}% | {dist[1]:6.1f}% | {dist[2]:6.1f}% | {dist[3]:6.1f}% | {dist[4]:6.1f}% | {dist[5]:6.1f}% | {dist[6]:6.1f}%")

In [ ]:
# Run inference on selected samples for all checkpoints
print(f"Running inference on {len(sample_indices)} samples across {len(checkpoint_files)} checkpoints...\\n")

# Store results: results[checkpoint_name][sample_idx] = {'pre', 'post', 'gt', 'pred'}
all_results = {}

for ckpt_path in checkpoint_files:
    ckpt_name = ckpt_path.stem.replace('unet_model_', '')  # e.g., "20260626_022648"
    
    try:
        checkpoint = torch.load(ckpt_path, map_location=device)
        
        # Load model
        model = UNet(in_channels=8, out_channels=7, bilinear=True).to(device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        # Load normalization stats
        channel_means = checkpoint['normalization']['channel_mean'].to(device)
        channel_stds = checkpoint['normalization']['channel_std'].to(device)
        
        # Run inference
        results = {}
        with torch.no_grad():
            for sample_idx in sample_indices:
                pre_img = pre_rgbn_tensor[sample_idx].float().unsqueeze(0).to(device)
                post_img = post_rgbn_tensor[sample_idx].float().unsqueeze(0).to(device)
                
                image = torch.cat([pre_img, post_img], dim=1)
                
                mean = channel_means.view(1, 8, 1, 1)
                std = channel_stds.view(1, 8, 1, 1)
                image = (image - mean) / (std + 1e-8)
                
                logits = model(image)
                predictions = logits.argmax(dim=1).squeeze(0)
                
                N = len(pre_rgbn_tensor)
                ground_truth = labels_tensor[N + sample_idx]
                
                results[sample_idx] = {
                    'pre_img': pre_rgbn_tensor[sample_idx],
                    'post_img': post_rgbn_tensor[sample_idx],
                    'ground_truth': ground_truth,
                    'predictions': predictions,
                }
        
        all_results[ckpt_name] = results
        print(f"✓ {ckpt_name}: inference complete")
        
    except Exception as e:
        print(f"❌ {ckpt_name}: {str(e)}")

print(f"\\n✓ Inference complete for {len(all_results)} checkpoints")

In [ ]:
# Create comparison visualization: GT + predictions from each checkpoint
# Layout: 3 rows (one per sample), columns: GT + one per checkpoint
num_cols = 1 + len(all_results)  # GT + predictions for each checkpoint
num_rows = len(sample_indices)

fig, axes = plt.subplots(num_rows, num_cols, figsize=(4 * num_cols, 4 * num_rows))

if num_rows == 1:
    axes = axes.reshape(1, -1)

for row, sample_idx in enumerate(sample_indices):
    # Get first result to extract images (same for all checkpoints)
    first_ckpt = list(all_results.keys())[0]
    data = all_results[first_ckpt][sample_idx]
    
    # Ground truth
    gt_labels = data['ground_truth'].numpy()
    gt_colored = np.zeros((512, 512, 3))
    for class_id in range(7):
        mask = gt_labels == class_id
        gt_colored[mask] = CLASS_COLORS[class_id]
    
    axes[row, 0].imshow(gt_colored)
    axes[row, 0].set_title(f"Sample {sample_idx}\\nGround Truth", fontsize=11, fontweight='bold')
    axes[row, 0].axis('off')
    
    # Predictions from each checkpoint
    for col, ckpt_name in enumerate(sorted(all_results.keys())):
        data = all_results[ckpt_name][sample_idx]
        pred_labels = data['predictions'].cpu().numpy()
        pred_colored = np.zeros((512, 512, 3))
        for class_id in range(7):
            mask = pred_labels == class_id
            pred_colored[mask] = CLASS_COLORS[class_id]
        
        axes[row, col + 1].imshow(pred_colored)
        axes[row, col + 1].set_title(f"{ckpt_name}\\nPredicted", fontsize=10, fontweight='bold')
        axes[row, col + 1].axis('off')

plt.tight_layout()
plt.savefig('/content/visualization_all_checkpoints.png', dpi=100, bbox_inches='tight')
print(f"✓ Comparison visualization saved to /content/visualization_all_checkpoints.png")
plt.show()

In [ ]:
# Create visualization
fig, axes = plt.subplots(len(sample_indices), 4, figsize=(16, 4 * len(sample_indices)))

if len(sample_indices) == 1:
    axes = axes.reshape(1, -1)

for row, sample_idx in enumerate(sample_indices):
    data = results[sample_idx]
    
    # Pre-fire RGB (bands 2, 1, 0 for BGR → RGB)
    pre_rgb = data['pre_img'][[2, 1, 0], :, :].numpy()  # [3, 512, 512]
    pre_rgb = np.transpose(pre_rgb, (1, 2, 0))  # [512, 512, 3]
    # Use 2nd-98th percentile for contrast stretch
    p2, p98 = np.percentile(pre_rgb, (2, 98))
    pre_rgb = np.clip((pre_rgb - p2) / (p98 - p2 + 1e-8), 0, 1)
    
    # Post-fire RGB
    post_rgb = data['post_img'][[2, 1, 0], :, :].numpy()  # [3, 512, 512]
    post_rgb = np.transpose(post_rgb, (1, 2, 0))  # [512, 512, 3]
    # Use 2nd-98th percentile for contrast stretch
    p2, p98 = np.percentile(post_rgb, (2, 98))
    post_rgb = np.clip((post_rgb - p2) / (p98 - p2 + 1e-8), 0, 1)
    
    # Ground truth labels (colored)
    gt_labels = data['ground_truth'].numpy()  # [512, 512]
    gt_colored = np.zeros((512, 512, 3))
    for class_id in range(7):
        mask = gt_labels == class_id
        gt_colored[mask] = CLASS_COLORS[class_id]
    
    # Predicted labels (colored)
    pred_labels = data['predictions'].cpu().numpy()  # [512, 512]
    pred_colored = np.zeros((512, 512, 3))
    for class_id in range(7):
        mask = pred_labels == class_id
        pred_colored[mask] = CLASS_COLORS[class_id]
    
    # Display
    axes[row, 0].imshow(pre_rgb)
    axes[row, 0].set_title(f"Sample {sample_idx}\nPre-fire RGB", fontsize=12, fontweight='bold')
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(post_rgb)
    axes[row, 1].set_title(f"Post-fire RGB", fontsize=12, fontweight='bold')
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(gt_colored)
    axes[row, 2].set_title(f"Ground Truth", fontsize=12, fontweight='bold')
    axes[row, 2].axis('off')
    
    axes[row, 3].imshow(pred_colored)
    axes[row, 3].set_title(f"Predicted", fontsize=12, fontweight='bold')
    axes[row, 3].axis('off')

plt.tight_layout()
plt.savefig('/content/visualization_iteration6.png', dpi=100, bbox_inches='tight')
print(f"✓ Visualization saved to /content/visualization_iteration6.png")
plt.show()

In [ ]:
# Create legend for color scheme
fig, ax = plt.subplots(figsize=(8, 6))
ax.axis('off')

# Create color legend
patches = []
labels = []

for class_id in range(7):
    color = CLASS_COLORS[class_id]
    name = CLASS_NAMES[class_id]
    patches.append(mpatches.Patch(facecolor=color, edgecolor='black', label=name))
    labels.append(name)

ax.legend(handles=patches, loc='center', fontsize=16, title='Burn Severity Classes', title_fontsize=18)
ax.set_title('Color Scheme for Burn Severity Classification', fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('/content/color_legend.png', dpi=100, bbox_inches='tight')
print(f"✓ Color legend saved to /content/color_legend.png")
plt.show()

print(f"\n" + "="*70)
print(f"BURN SEVERITY COLOR SCHEME")
print(f"="*70)
for class_id in range(7):
    color = CLASS_COLORS[class_id]
    name = CLASS_NAMES[class_id]
    print(f"  Class {class_id}: {name:15s} RGB({color[0]:.2f}, {color[1]:.2f}, {color[2]:.2f})")
print(f"="*70)